# 讀入資料與整理

拿到考試資料後的標準流程。每一步都可以直接複製程式碼來改。

| 步驟 | 做什麼 | 主要寫法 |
|---|---|---|
| 1. 速覽 | 有哪些工作表、幾列幾欄、欄位長怎樣 | `sk.peek('檔名.xlsx')` |
| 2. 讀檔 | 選對工作表、跳過標題列、指定編碼 | `pd.read_excel(..., sheet_name=, header=)`、`pd.read_csv(..., encoding=)` |
| 3. 檢查 | 大小、型態、缺值、重複、數值範圍 | `df.shape`、`df.dtypes`、`df.isna().sum()`、`df.describe()` |
| 4. 欄位名稱 | 去掉空白、改成好打的名字 | `df.columns.str.strip()`、`df.rename(columns=...)` |
| 5. 型態轉換 | 文字變數字、代碼變類別、日期 | `pd.to_numeric`、`.map(...)`、`pd.to_datetime` |
| 6. 缺值與重複 | 先算有多少，再依題目決定怎麼處理 | `dropna`、`drop_duplicates` |
| 7. 篩選與挑欄 | 只留題目要的資料 | `df[df['欄'] == 值]`、`df[['a', 'b']]` |
| 8. 新欄位與彙總 | 算比例、分組統計、交叉表 | `groupby().agg()`、`value_counts()`、`pd.crosstab` |
| 9. 分析 | 敘述統計、畫圖 | `sk.describe`、`sk.freq_table`、各章筆記 |

> **資料要放哪裡**：老師的考卷如果寫 `pd.read_excel('檔名.xlsx')`（沒有路徑），資料要跟考卷放在**同一個資料夾**（`exam/`；用 `5_匯入考卷` 匯入時會自動一起複製）。自己練習時資料放 `data/`，在 `exam/` 裡的 notebook 要寫 `'../data/檔名.xlsx'`。

In [ ]:
import statistics

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import statkit as sk
sk.setup()

## 0. 準備範例資料（考試時跳過這格）

下面這格會做出一份「故意弄髒」的範例資料，模擬考試資料常見的問題：
第一列是標題、欄位名稱有空白、數字帶千分位逗號、地區用代碼、有缺值、有重複列，另外還有一份 Big5 編碼的 CSV。

In [ ]:
import tempfile
from pathlib import Path

folder = Path(tempfile.gettempdir()) / 'statkit_demo'
folder.mkdir(exist_ok=True)
xlsx_path = folder / '範例資料.xlsx'
csv_path = folder / '範例資料_big5.csv'

raw = pd.DataFrame({
    ' Food ': [8909, 5684, 10706, 7310, None, 7310, 9120],
    'Income': ['56,697', '35,945', '52,687', '41,203', '48,000', '41,203', 'n/a'],
    'Region': [1, 2, 3, 4, 1, 4, 2],
    'Location': [1, 2, 1, 1, 2, 1, 2],
    '調查日期': ['2026/03/01', '2026/03/02', '2026/03/02', '2026/03/05', '2026/03/06', '2026/03/05', '2026/03/08'],
})
with pd.ExcelWriter(xlsx_path) as w:
    pd.DataFrame([['家庭消費調查（單位：美元）']]).to_excel(w, sheet_name='Consumer Food', index=False, header=False)
    raw.to_excel(w, sheet_name='Consumer Food', index=False, startrow=1)       # 第 1 列是標題，第 2 列才是欄位名稱
    pd.DataFrame({'說明': ['Region: 1=東北, 2=中西, 3=南, 4=西']}).to_excel(w, sheet_name='說明', index=False)
raw.to_csv(csv_path, index=False, encoding='cp950')
print(xlsx_path)

---
## 1. 速覽：先看檔案裡有什麼

`sk.peek` 會列出所有工作表、每欄的型態、缺值、範例值和前幾列，並**印出讀檔的程式碼**。

In [ ]:
df = sk.peek(xlsx_path)        # 考試時：df = sk.peek('檔名.xlsx')（資料跟考卷在同一個資料夾）

看到欄位名稱是 `Unnamed: 0`、第一列是標題文字 → 代表**欄位名稱不在第一列**，讀檔時要加 `header=1`（第 2 列才是欄位名稱）。

---
## 2. 讀檔

In [ ]:
df = pd.read_excel(xlsx_path, sheet_name='Consumer Food', header=1)   # header=1：第 2 列當欄位名稱
df

| 參數 | 用途 | 例子 |
|---|---|---|
| `sheet_name` | 工作表名稱或編號（0 = 第一個） | `sheet_name='Consumer Food'` |
| `header` | 第幾列是欄位名稱（從 0 數） | `header=1` |
| `skiprows` | 跳過前幾列 | `skiprows=2` |
| `usecols` | 只讀某些欄 | `usecols=['Food', 'Income']` 或 `usecols='A:C'` |
| `nrows` | 只讀前幾列 | `nrows=100` |
| `sheet_name=None` | 一次讀所有工作表（得到 dict） | `sheets = pd.read_excel(path, sheet_name=None)` |

**CSV**：中文亂碼或出現 `UnicodeDecodeError`，改用 `encoding='cp950'`（Big5）或 `'utf-8-sig'`；`sk.peek` 會自動試。

In [ ]:
df_csv = pd.read_csv(csv_path, encoding='cp950')
df_csv.head(3)

---
## 3. 檢查：大小、型態、缺值、重複

In [ ]:
print('列數、欄數：', df.shape)
print()
print(df.dtypes)          # Income 是 str / object（文字），代表裡面有逗號或不是數字的東西

In [ ]:
print('每欄缺值數：')
print(df.isna().sum())
print()
print('完全重複的列數：', df.duplicated().sum())
print('每欄不同值的個數：')
print(df.nunique())

In [ ]:
df.describe()             # 只會列出數值欄位；最小值、最大值不合理（例如負的收入）要特別注意

---
## 4. 整理欄位名稱

欄位名稱前後有空白（`' Food '`）會讓 `df['Food']` 找不到。

In [ ]:
df.columns = df.columns.str.strip()                       # 去掉前後空白
df = df.rename(columns={'調查日期': 'Date'})               # 改成好打的名字
list(df.columns)

---
## 5. 型態轉換

In [ ]:
# 文字數字：去掉千分位逗號再轉數字；轉不了的（例如 'n/a'）變成 NaN
df['Income'] = pd.to_numeric(df['Income'].str.replace(',', ''), errors='coerce')

# 日期
df['Date'] = pd.to_datetime(df['Date'])

# 代碼 → 名稱（題目會說明代碼的意思，例如 HW03 的 Region、Location）
region_names = {1: 'North East', 2: 'Middle West', 3: 'South', 4: 'West'}
df['RegionName'] = df['Region'].map(region_names)
df['Metro'] = df['Location'].map({1: 'inside', 2: 'outside'})

df.dtypes

其他常用：百分比文字 `'35%'` → `df['p'].str.rstrip('%').astype(float) / 100`；
四捨五入 `df['x'].round(2)`；整數 `df['x'].astype(int)`（有缺值時不能轉 int）。

---
## 6. 缺值與重複列

**先算有多少、再依題目處理**，並在答案裡寫清楚怎麼處理的（例如「刪除 1 筆 Food 缺值的資料，剩 5 筆」）。題目沒要求就不要自己補值。

In [ ]:
print('處理前：', df.shape)
print(df.isna().sum()[lambda s: s > 0])        # 只列出有缺值的欄位

df = df.drop_duplicates()                      # 刪除完全重複的列
df = df.dropna(subset=['Food', 'Income'])      # 只刪除這兩欄有缺值的列
print('處理後：', df.shape)

| 寫法 | 意思 |
|---|---|
| `df.dropna()` | 刪掉任何一欄有缺值的列 |
| `df.dropna(subset=['a'])` | 只看 a 欄 |
| `df['a'].fillna(0)` | 缺值補 0（題目要求時才用） |
| `df['a'].fillna(df['a'].mean())` | 缺值補平均數 |
| `df['a'].mean()` | pandas 的統計函式本來就會**自動略過缺值** |
| `df.drop_duplicates(subset=['id'])` | 依某欄判斷重複 |

---
## 7. 篩選與挑欄

In [ ]:
food = df['Food']                                   # 一欄（Series）
sub = df[['Food', 'Income']]                        # 多欄（DataFrame）

north_east = df[df['Region'] == 1]                  # 條件篩選
rich_metro = df[(df['Income'] > 45000) & (df['Location'] == 1)]   # 且：&（每個條件都要加括號）
west_or_south = df[df['Region'].isin([3, 4])]       # 屬於某幾個值
mid_income = df[df['Income'].between(40000, 55000)] # 介於（含兩端）
march_early = df[df['Date'] <= '2026-03-05']        # 日期也可以比較

rich_metro

- 或：`|`、非：`~`，例如 `df[(df['Region'] == 1) | (df['Region'] == 2)]`、`df[~df['Region'].isin([1])]`
- 依位置取：`df.iloc[0]`（第一列）、`df.iloc[:10]`（前 10 列）；依標籤取：`df.loc[df['Food'] > 8000, 'Income']`
- 排序：`df.sort_values('Income', ascending=False)`；前幾大：`df.nlargest(3, 'Income')`

---
## 8. 新欄位與彙總

更多「多個變數」的寫法（分組統計、樞紐分析表、交叉表、相關矩陣）見 `02_敘述統計.ipynb` 的「多個變數一起分析」；畫圖見 `01_視覺化.ipynb` 的「多個變數一起畫」。

In [ ]:
df['FoodShare'] = df['Food'] / df['Income']         # 新欄位：食物支出占收入比例

# 分組統計
display(df.groupby('RegionName')['Income'].agg(['count', 'mean', 'median', 'std']).round(2))

# 類別的次數與比例
display(df['Metro'].value_counts())
display(df['Metro'].value_counts(normalize=True).round(4))

# 交叉表（兩個類別變數）
display(pd.crosstab(df['RegionName'], df['Metro'], margins=True))

- 樞紐分析表：`pd.pivot_table(df, values='Income', index='RegionName', columns='Metro', aggfunc='mean')`
- 依數值分組（例如收入分級）：`pd.cut(df['Income'], bins=[0, 40000, 50000, 100000], right=False)`
- 每組多個統計量：`df.groupby('Region').agg(平均收入=('Income', 'mean'), 筆數=('Food', 'count'))`

---
## 日期與依月份彙總（趨勢圖）

`pd.to_datetime` 把文字轉成日期，`dt.to_period('M')` 取年月，再 `groupby` 加總或平均。

In [ ]:
sales = pd.DataFrame({'Date': ['2025-01-05', '2025-01-20', '2025-02-03', '2025-02-17', '2025-03-08', '2025-03-30'],
                      'Revenue': [120, 135, 98, 143, 160, 151]})
sales['Date'] = pd.to_datetime(sales['Date'])                                  # 文字 → 日期
monthly = sales.groupby(sales['Date'].dt.to_period('M'))['Revenue'].sum()     # 依月份加總（.mean() 平均）
print(monthly)
monthly.index = monthly.index.astype(str)
plt.plot(monthly.index, monthly.values, marker='o', color='black')
plt.ylabel('Revenue')
plt.title('Monthly revenue')
plt.show()

---
## 9. 開始分析

資料整理好之後，就照題目用各章筆記的方法：

In [ ]:
sk.describe(df['Income'])

In [ ]:
sk.box({name: g['Income'] for name, g in df.groupby('RegionName')}, ylabel='Income ($)', title='各地區收入')
plt.show()

- 敘述統計、次數分配、離群值 → [02_敘述統計](02_敘述統計.ipynb)
- 直方圖、長條圖、散佈圖、盒鬚圖 → [01_視覺化](01_視覺化.ipynb)
- 用資料估計機率、跟分布比較 → [03_機率與離散分布](03_機率與離散分布.ipynb)、[05_常態與模擬](05_常態與模擬.ipynb)

---
## 常見陷阱

| 狀況 | 看到的現象 | 解法 |
|---|---|---|
| 標題列不在第一列 | 欄位名稱是 `Unnamed: 0`、第一列是說明文字 | `header=1` 或 `skiprows=1` |
| 欄位名稱有空白 | `KeyError: 'Food'` | `df.columns = df.columns.str.strip()`，或先 `print(list(df.columns))` 看實際名稱 |
| 數字被讀成文字 | `dtypes` 是 `str` 或 `object`、`describe()` 沒有這欄 | 去掉逗號、`$`、`%` 後 `pd.to_numeric(..., errors='coerce')` |
| 合併儲存格 | 只有第一列有值，下面都是 NaN | `df['欄'] = df['欄'].ffill()`（往下填） |
| 中文 CSV 亂碼 | 亂碼或 `UnicodeDecodeError` | `encoding='cp950'` 或 `'utf-8-sig'` |
| Excel 日期變成數字 | 例如 46082 | `pd.to_datetime(df['d'], unit='D', origin='1899-12-30')` |
| 單位 | 題目說「千元」「百萬」 | 計算前後記得換算，答案寫清楚單位 |
| 代碼當成數字算 | 對 Region 算平均數 | 代碼是類別，用 `value_counts()`、`groupby`，不要算平均 |
| 改到原始資料 | 後面的題目結果跟著變 | 每題需要不同處理時，先 `d = df.copy()` 再改 |

---
## 答案裡怎麼描述資料（範例）

> 資料共有 200 筆家庭資料、5 個欄位（Food、Income、Debt、Region、Location），一筆資料代表一個家庭。
> 其中 Food、Income、Debt 為數值變數（單位：美元），Region（1–4）與 Location（1、2）為類別代碼。
> Income 有 1 筆缺值，分析時刪除該筆，剩 199 筆。